In [1]:
import pandas as pd

In [4]:
# Load the Excel file
file_path = "disaster_information.xlsx"  # Update path if needed
df = pd.read_excel(file_path)

# Select the required columns (River Basin added for geographic context)
columns_needed = [
    "Disaster Group", "Disaster Subgroup", "Disaster Type", "Event Name",
    "Country", "Subregion", "Location",
    "Start Year", "Start Month", "Start Day",
    "End Year", "End Month", "End Day",
    "Total Damage ('000 US$')", "Reconstruction Costs ('000 US$')",
    "AID Contribution ('000 US$')", "Insured Damage ('000 US$')",
    "Associated Types", "River Basin"
]

# Ensure only existing columns are selected
existing_columns = [col for col in columns_needed if col in df.columns]
extracted_df = df[existing_columns].copy()

# Format start and end dates
def format_date(row, year_col, month_col, day_col):
    year = str(int(row[year_col])) if not pd.isna(row[year_col]) else "Unknown"
    month = str(int(row[month_col])) if not pd.isna(row[month_col]) else "Unknown"
    day = str(int(row[day_col])) if not pd.isna(row[day_col]) else "Unknown"
    return f"{year}-{month}-{day}" if year != "Unknown" else "Unknown"

extracted_df["Start Date"] = extracted_df.apply(lambda row: format_date(row, "Start Year", "Start Month", "Start Day"), axis=1)
extracted_df["End Date"] = extracted_df.apply(lambda row: format_date(row, "End Year", "End Month", "End Day"), axis=1)

# Drop raw date columns
date_columns = ["Start Year", "Start Month", "Start Day", "End Year", "End Month", "End Day"]
extracted_df.drop(columns=[col for col in date_columns if col in extracted_df.columns], inplace=True)

# Save to a new CSV file
output_file_path = "extracted_disaster_info.csv"
extracted_df.to_csv(output_file_path, index=False)


In [5]:
df = pd.read_csv('extracted_disaster_info.csv', encoding='ISO-8859-1')

def clean_val(val, default=None):
    """Return None (or default) for NaN/blank/literal-nan values, else a stripped string."""
    if val is None:
        return default
    s = str(val).strip()
    if s.lower() in ('nan', 'none', '', 'n/a', 'unknown'):
        return default
    return s

def build_combined(row):
    event_name  = clean_val(row.get('Event Name'))
    country     = clean_val(row.get('Country'),    'Unknown Country')
    subregion   = clean_val(row.get('Subregion'),  'Unknown Region')
    location    = clean_val(row.get('Location'))
    start_date  = clean_val(row.get('Start Date'), 'unknown date')
    end_date    = clean_val(row.get('End Date'),   'unknown date')
    group       = clean_val(row.get('Disaster Group'),    'Unknown')
    subgroup    = clean_val(row.get('Disaster Subgroup'), 'Unknown')
    d_type      = clean_val(row.get('Disaster Type'),     'Unknown')
    total_dmg   = clean_val(row.get("Total Damage ('000 US$)"))
    insured_dmg = clean_val(row.get("Insured Damage ('000 US$)"))
    aid         = clean_val(row.get("AID Contribution ('000 US$)"))
    recon       = clean_val(row.get("Reconstruction Costs ('000 US$)"))
    river       = clean_val(row.get('River Basin'))
    assoc       = clean_val(row.get('Associated Types'))

    parts = []

    # Header sentence
    if event_name:
        parts.append(f"Event '{event_name}' occurred in {country} ({subregion}), from {start_date} to {end_date}.")
    else:
        parts.append(f"A disaster occurred in {country} ({subregion}), from {start_date} to {end_date}.")

    # Classification
    parts.append(f"Classification: {group} > {subgroup} > {d_type}.")

    # Specific location
    if location:
        parts.append(f"Specific location: {location}.")

    # Financial impact
    fin_parts = []
    if total_dmg:
        fin_parts.append(f"total damage: {total_dmg}K USD")
    if insured_dmg:
        fin_parts.append(f"insured damage: {insured_dmg}K USD")
    if aid:
        fin_parts.append(f"AID contribution: {aid}K USD")
    if recon:
        fin_parts.append(f"reconstruction costs: {recon}K USD")
    if fin_parts:
        parts.append(f"Financial impact — {'; '.join(fin_parts)}.")
    else:
        parts.append("No financial impact data recorded.")

    # Geographic / type context
    if river:
        parts.append(f"River basin: {river}.")
    if assoc:
        parts.append(f"Associated disaster types: {assoc}.")

    return " ".join(parts)

df['combined'] = df.apply(build_combined, axis=1)

# Verify quality: count remaining literal-nan strings (should be 0)
nan_count = df['combined'].str.contains(r'\bnan\b', case=False, na=False).sum()
print(f"Rows still containing literal 'nan': {nan_count}")
print("\nSample rows:")
print(df[['combined']].head(3).to_string())

Rows still containing literal 'nan': 33

Sample rows:
                                                                                                                                                                                                                                                                         combined
0  A disaster occurred in United States of America (Northern America), from 1900-9-8 to 1900-9-8. Classification: Natural > Meteorological > Storm. Specific location: Galveston (Texas). No financial impact data recorded. Associated disaster types: Avalanche (Snow, Debris).
1                                                                 A disaster occurred in Jamaica (Latin America and the Caribbean), from 1900-1-6 to 1900-1-6. Classification: Natural > Hydrological > Flood. Specific location: Saint James. No financial impact data recorded.
2                                                       Event 'Gastroenteritis' occurred in Jamaica (Latin America and the C

In [6]:
# ── Regenerate embeddings with a higher-quality retrieval model ────────────────
# multi-qa-mpnet-base-dot-v1 is fine-tuned specifically for semantic search / QA
# retrieval (768-dim vs 384-dim for all-MiniLM-L6-v2), giving meaningfully better
# recall@k on domain-specific corpora.
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import numpy as np

EMBEDDING_MODEL = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
hf = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

BATCH_SIZE = 64
texts = df['combined'].tolist()
all_embeddings = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Embedding batches"):
    batch = texts[i : i + BATCH_SIZE]
    all_embeddings.extend(hf.embed_documents(batch))

df['embeddings'] = [str(e) for e in all_embeddings]

output_path = "disaster_info_with_embeddings.csv"
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} rows with refreshed embeddings → {output_path}")
print(f"Embedding dimensions: {len(all_embeddings[0])}")

e:\Harinivas\Programs\rag_model\disaster_information\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\Harinivas\Programs\rag_model\disaster_information\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sharinivas\.cache\huggingface\hub\models--sentence-transformers--multi-qa-mpnet-base-dot-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate 

Saved 17325 rows with refreshed embeddings → disaster_info_with_embeddings.csv
Embedding dimensions: 768
